# TLC Yearly Budget Forecast (FY2027-FY2031)

A simple, honest trend forecast for TLC's Adopted, Modified, and Actual
budget totals, built the same way as `dot_forecast.ipynb`: rebuilt directly
from the raw CSVs (not imported from `tlc_analysis.ipynb`), applying the two
corrections documented there --

- **No agency filter needed** -- both TLC files are single-agency across
  every fiscal year (confirmed in `tlc_analysis.ipynb` Step 1).
- **Scope filter on spending**: keep only spending rows whose `Budget_Code`
  (leading token) exists in the TLC budget file's set of codes -- drops 618
  rows with a missing/non-applicable code and no budget line to reconcile
  against (`tlc_analysis.ipynb` Step 3).

**Headline limitation, stated up front and repeated throughout:** this is
~10 annual observations (FY2017-2026). That supports a simple linear trend
with an honest uncertainty band -- not a complex model. Every forecast below
ships with a 95% prediction interval, never a bare point estimate.

**FY2023 note:** FY2023's Adopted/Modified budget (~\$155-162M) is roughly
3x every neighboring year, driven by one-time medallion-relief
appropriations (`MLG2` Medallion Loan Guarantee + `CR02` Medallion Relief
Fund, \$50M each) tied to the NYC taxi medallion debt crisis -- a real
budget event, not a data error (see `tlc_analysis.ipynb` Step 6). **It is
kept in the training data** (unlike FY2027, which is dropped for being a
partial year) because it's genuine agency spending history, not an
artifact -- but it does mean the fitted trend is pulled upward by one
unusual year, discussed again in Step 6.

In [26]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LinearRegression

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 130)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = "raw datasets"
BUDGET_PATH = f"{DATA_DIR}/TCL_Budget_2017_2027.csv"
SPENDING_PATH = f"{DATA_DIR}/TCL_Spending_Clean_2017_2027.csv" 

## Step 1: Load & Correct

Rebuild the corrected year-level table from scratch: TLC budget as-is (no
agency filter needed) joined with scope-filtered spending (`Budget_Code`
leading token must exist in the TLC budget file's code set).

In [27]:
# --- Budget: single-agency already, no filter needed ---
budget_raw = pd.read_csv(BUDGET_PATH)
budget = budget_raw.copy()
print(f"Budget: {len(budget_raw):,} rows, Agency values: {budget['Agency'].unique()}")

# --- Spending: scope-filtered (Budget Code must exist in the TLC budget file) ---
spending_raw = pd.read_csv(SPENDING_PATH)
tlc_budget_codes = set(budget["Budget Code"].astype(str).str.strip())
spending_code_lead = spending_raw["Budget_Code"].astype(str).str.extract(r"^([^\s(]+)")[0]
is_in_scope = spending_code_lead.isin(tlc_budget_codes)
spending = spending_raw[is_in_scope].copy()
print(f"Spending: {len(spending_raw):,} raw rows -> {len(spending):,} scope-filtered rows "
      f"(dropped {len(spending_raw) - len(spending):,} missing/non-applicable-code rows)")

Budget: 3,328 rows, Agency values: ['NYC Taxi and Limousine Commission']
Spending: 18,375 raw rows -> 17,757 scope-filtered rows (dropped 618 missing/non-applicable-code rows)


In [28]:
# --- Aggregate to fiscal year, FY2017-2027 ---
budget_by_year = (
    budget[budget["Year"].between(2017, 2027)]
    .groupby("Year")[["Adopted", "Modified"]]
    .sum()
    .rename(columns={"Adopted": "adopted", "Modified": "modified"})
)
spending_by_year = (
    spending[spending["Fiscal_year"].between(2017, 2027)]
    .groupby("Fiscal_year")["Check_Amount"]
    .sum()
    .rename("actual")
)

full_table = (
    budget_by_year.join(spending_by_year, how="outer")
    .rename_axis("fiscal_year")
    .reset_index()
    .sort_values("fiscal_year")
    .reset_index(drop=True)
)
print("Full corrected table, FY2017-2027:")
full_table

Full corrected table, FY2017-2027:


,fiscal_year,adopted,modified,actual
0,2017,70612081,46930977,"40,380,013.85"
1,2018,57479441,48964179,"41,550,268.55"
2,2019,52514485,49317576,"43,346,568.80"
3,2020,53235198,53508669,"47,401,658.05"
4,2021,54115393,53925677,"47,959,631.36"
5,2022,55474235,66077398,"59,909,548.61"
6,2023,155512440,162372128,"115,098,150.42"
7,2024,60328172,59824556,"58,870,172.75"
8,2025,60317295,55681391,"48,576,817.42"
9,2026,58133858,61806067,"53,970,000.24"


FY2027 is excluded from the training set for the same reason as DOT's
FY2027: it's a partial fiscal year (barely started as of this analysis --
today is 2026-08-10, a few weeks into FY2027), so `actual` is nowhere near a
comparable full-year total. It's held out, unused in fitting, for the Step
5 sanity check on `adopted` (which is set before the fiscal year starts and
so is already a known, complete figure for FY2027).

In [29]:
train_table = full_table[full_table["fiscal_year"] <= 2026].reset_index(drop=True)
fy2027_row = full_table[full_table["fiscal_year"] == 2027].iloc[0]

print(f"Training table: FY2017-2026 ({len(train_table)} observations)")
print("FY2027 (excluded from training, held out for Step 5 sanity check):")
print(fy2027_row)
print()
train_table

Training table: FY2017-2026 (10 observations)
FY2027 (excluded from training, held out for Step 5 sanity check):
fiscal_year        2,027.00
adopted       69,723,377.00
modified      69,723,377.00
actual         6,471,393.58
Name: 10, dtype: float64



,fiscal_year,adopted,modified,actual
0,2017,70612081,46930977,"40,380,013.85"
1,2018,57479441,48964179,"41,550,268.55"
2,2019,52514485,49317576,"43,346,568.80"
3,2020,53235198,53508669,"47,401,658.05"
4,2021,54115393,53925677,"47,959,631.36"
5,2022,55474235,66077398,"59,909,548.61"
6,2023,155512440,162372128,"115,098,150.42"
7,2024,60328172,59824556,"58,870,172.75"
8,2025,60317295,55681391,"48,576,817.42"
9,2026,58133858,61806067,"53,970,000.24"


## Step 2: Visualize

Adopted, Modified, and Actual over FY2017-2026. FY2023's medallion-relief
spike is visible as a real bump in Adopted/Modified, not smoothed away.

In [30]:
SURFACE = "#fcfcfb"
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED = "#898781"
GRIDLINE = "#e1e0d9"
BASELINE = "#c3c2b7"
SERIES = {
    "adopted": "#2a78d6",   # categorical slot 1: blue
    "modified": "#eb6834",  # categorical slot 2: orange
    "actual": "#1baf7a",    # categorical slot 3: aqua
}

fig, ax = plt.subplots(figsize=(8, 5.5), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

for col, color in SERIES.items():
    ax.plot(
        train_table["fiscal_year"], train_table[col] / 1e6,
        color=color, linewidth=2, marker="o", markersize=8,
        markerfacecolor=color, markeredgecolor=SURFACE, markeredgewidth=2,
        label=col.capitalize(),
    )

ax.annotate(
    "FY2023: medallion relief\n(MLG2 + CR02, one-time)",
    xy=(2023, train_table.loc[train_table["fiscal_year"] == 2023, "modified"].iloc[0] / 1e6),
    xytext=(2023.3, 130), fontsize=9, color=INK_SECONDARY,
    arrowprops=dict(arrowstyle="->", color=INK_MUTED, lw=1),
)

ax.set_xlabel("Fiscal Year", color=INK_SECONDARY)
ax.set_ylabel("$ Millions", color=INK_SECONDARY)
ax.set_title(
    "TLC Budget: Adopted vs. Modified vs. Actual (FY2017-2026)\n"
    "FY2027 excluded -- partial year",
    color=INK_PRIMARY, fontsize=12, pad=12,
)
ax.set_xticks(train_table["fiscal_year"])

ax.grid(True, color=GRIDLINE, linewidth=1, linestyle="-")
ax.set_axisbelow(True)
for spine_name, spine in ax.spines.items():
    if spine_name in ("top", "right"):
        spine.set_visible(False)
    else:
        spine.set_color(BASELINE)
ax.tick_params(colors=INK_MUTED)

ax.legend(frameon=False, loc="upper left", labelcolor=INK_SECONDARY)

fig.tight_layout()

TREND_PLOT_PATH = "tlc_forecast_trend.png"
fig.savefig(TREND_PLOT_PATH, dpi=150, facecolor=SURFACE)
print(f"Saved trend plot to {TREND_PLOT_PATH}")
plt.show()

Saved trend plot to tlc_forecast_trend.png


/var/folders/6v/95zgdc6s2w71rmzcwm6mww_40000gn/T/ipykernel_45860/2105644118.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 3: Forecast

Simple linear regression of each series on Fiscal Year: `y = slope * Year +
intercept`. Kept deliberately simple -- with 10 points, a more flexible
model would fit noise, not signal.

In [31]:
def fit_and_forecast(df, y_col, forecast_years, alpha=0.05):
    '''OLS trend fit + classic prediction intervals for new-year forecasts.

    Accounts for residual variance AND the extra uncertainty of predicting a
    new point away from the data's center -- the wider, more honest interval
    for a forecast rather than a fitted-mean estimate.
    '''
    x = df["fiscal_year"].values.astype(float)
    y = df[y_col].values.astype(float)
    n = len(x)

    model = LinearRegression().fit(x.reshape(-1, 1), y)
    slope = model.coef_[0]
    intercept = model.intercept_
    r2 = model.score(x.reshape(-1, 1), y)

    y_pred = model.predict(x.reshape(-1, 1))
    resid = y - y_pred
    dof = n - 2
    resid_se = np.sqrt(np.sum(resid ** 2) / dof)
    xbar = x.mean()
    sxx = np.sum((x - xbar) ** 2)
    t_crit = stats.t.ppf(1 - alpha / 2, df=dof)

    forecast_rows = []
    for x0 in forecast_years:
        yhat0 = slope * x0 + intercept
        se_pred = resid_se * np.sqrt(1 + 1 / n + (x0 - xbar) ** 2 / sxx)
        margin = t_crit * se_pred
        forecast_rows.append({
            "fiscal_year": x0,
            "point_forecast": yhat0,
            "lower_95": yhat0 - margin,
            "upper_95": yhat0 + margin,
            "margin": margin,
        })

    return {
        "slope": slope, "intercept": intercept, "r2": r2, "n": n, "dof": dof,
        "residual_se": resid_se, "forecast": pd.DataFrame(forecast_rows),
    }

SERIES_COLS = ["adopted", "modified", "actual"]
FORECAST_YEARS = [2027, 2028]

fits = {col: fit_and_forecast(train_table, col, FORECAST_YEARS) for col in SERIES_COLS}

print("Trend fit summary (FY2017-2026, n=10):")
summary_rows = []
for col in SERIES_COLS:
    f = fits[col]
    summary_rows.append({
        "series": col, "slope ($/year)": f["slope"], "intercept": f["intercept"],
        "r2": round(f["r2"], 4), "residual_se": f["residual_se"],
    })
pd.DataFrame(summary_rows)

Trend fit summary (FY2017-2026, n=10):


,series,slope ($/year),intercept,r2,residual_se
0,adopted,"1,544,363.48","-3,054,158,512.57",0.02,"32,789,757.56"
1,modified,"3,467,716.92","-6,944,148,894.43",0.09,"34,805,610.00"
2,actual,"2,813,049.30","-5,630,872,867.27",0.15,"21,427,493.74"


In [32]:
print("FY2027-2028 point forecasts (95% prediction intervals):\n")
for col in SERIES_COLS:
    print(f"--- {col} ---")
    print(fits[col]["forecast"].to_string(index=False))
    print()

FY2027-2028 point forecasts (95% prediction intervals):

--- adopted ---
 fiscal_year  point_forecast       lower_95       upper_95        margin
        2027   76,266,258.93 -15,306,014.90 167,838,532.77 91,572,273.83
        2028   77,810,622.41 -18,195,103.04 173,816,347.86 96,005,725.45

--- modified ---
 fiscal_year  point_forecast       lower_95       upper_95         margin
        2027   84,913,304.87 -12,288,659.60 182,115,269.34  97,201,964.47
        2028   88,381,021.79 -13,526,954.54 190,288,998.11 101,907,976.32

--- actual ---
 fiscal_year  point_forecast      lower_95       upper_95        margin
        2027   71,178,054.13 11,337,284.63 131,018,823.63 59,840,769.50
        2028   73,991,103.42 11,253,156.16 136,729,050.69 62,737,947.27



## Step 4: Sanity Check -- FY2027 Adopted

FY2027's Adopted figure is already known (set before the fiscal year
starts) -- held out of training in Step 1 specifically to serve as a free,
honest out-of-sample check on the trend.

In [33]:
actual_fy2027_adopted = fy2027_row["adopted"]
forecast_fy2027 = fits["adopted"]["forecast"].set_index("fiscal_year").loc[2027]

error = forecast_fy2027["point_forecast"] - actual_fy2027_adopted
pct_error = error / actual_fy2027_adopted * 100
within_interval = forecast_fy2027["lower_95"] <= actual_fy2027_adopted <= forecast_fy2027["upper_95"]

print("FY2027 Adopted -- trend forecast vs. actual data:")
print(f"  Trend point forecast : {forecast_fy2027['point_forecast']:>15,.0f}")
print(f"  95% interval         : [{forecast_fy2027['lower_95']:,.0f}, {forecast_fy2027['upper_95']:,.0f}]")
print(f"  Actual (in the data) : {actual_fy2027_adopted:>15,.0f}")
print(f"  Error                : {error:>15,.0f}  ({pct_error:+.2f}%)")
print(f"  Falls within 95% PI  : {within_interval}")

FY2027 Adopted -- trend forecast vs. actual data:
  Trend point forecast :      76,266,259
  95% interval         : [-15,306,015, 167,838,533]
  Actual (in the data) :      69,723,377
  Error                :       6,542,882  (+9.38%)
  Falls within 95% PI  : True


## Step 5: Extended Forecast -- Next 5 Fiscal Years (FY2027-FY2031)

Today is 2026-08-10, a few weeks into FY2027 -- so the "next 5 years" from
here is **FY2027 through FY2031**. Reuses the exact same linear models fit
on FY2017-2026 in Step 3 (`fit_and_forecast` is deterministic OLS, so
calling it again with more forecast years re-derives the identical line and
only adds new projected years -- verified below, not just asserted).
FY2027's `adopted` value is shown as the trend forecast for continuity, but
the actually-adopted figure from Step 4 is the one to use in practice --
it's already known and printed alongside it.

In [34]:
FORECAST_HORIZON = [2027, 2028, 2029, 2030, 2031]

extended_fits = {col: fit_and_forecast(train_table, col, FORECAST_HORIZON) for col in SERIES_COLS}

print("Confirming the extended fit reuses the identical Step 3 model:")
for col in SERIES_COLS:
    slope_match = np.isclose(extended_fits[col]["slope"], fits[col]["slope"])
    intercept_match = np.isclose(extended_fits[col]["intercept"], fits[col]["intercept"])
    print(f"  {col:>9}: slope matches = {slope_match}, intercept matches = {intercept_match}")

Confirming the extended fit reuses the identical Step 3 model:
    adopted: slope matches = True, intercept matches = True
   modified: slope matches = True, intercept matches = True
     actual: slope matches = True, intercept matches = True


In [35]:
table_rows = []
for year in FORECAST_HORIZON:
    row = {"fiscal_year": year}
    for col in SERIES_COLS:
        fc = extended_fits[col]["forecast"].set_index("fiscal_year").loc[year]
        row[f"{col}_point"] = fc["point_forecast"]
        row[f"{col}_low95"] = fc["lower_95"]
        row[f"{col}_high95"] = fc["upper_95"]
    table_rows.append(row)

forecast_table = pd.DataFrame(table_rows)
print("FY2027-FY2031 point forecasts with 95% prediction intervals:")
forecast_table

FY2027-FY2031 point forecasts with 95% prediction intervals:


,fiscal_year,adopted_point,adopted_low95,adopted_high95,modified_point,modified_low95,modified_high95,actual_point,actual_low95,actual_high95
0,2027,"76,266,258.93","-15,306,014.90","167,838,532.77","84,913,304.87","-12,288,659.60","182,115,269.34","71,178,054.13","11,337,284.63","131,018,823.63"
1,2028,"77,810,622.41","-18,195,103.04","173,816,347.86","88,381,021.79","-13,526,954.54","190,288,998.11","73,991,103.42","11,253,156.16","136,729,050.69"
2,2029,"79,354,985.89","-21,577,270.10","180,287,241.88","91,848,738.71","-15,288,641.96","198,986,119.38","76,804,152.72","10,846,809.82","142,761,495.62"
3,2030,"80,899,349.37","-25,383,971.42","187,182,670.16","95,316,455.63","-17,500,963.21","208,133,874.47","79,617,202.01","10,163,038.28","149,071,365.75"
4,2031,"82,443,712.85","-29,554,373.09","194,441,798.78","98,784,172.55","-20,099,344.41","217,667,689.52","82,430,251.31","9,241,595.35","155,618,907.26"


In [36]:
print("Known FY2027 Adopted (actual, not trend) for reference: "
      f"{actual_fy2027_adopted:,.0f}")
print()
print("Interval width, FY2027 -> FY2031, by series (should widen monotonically):")
for col in SERIES_COLS:
    fc = extended_fits[col]["forecast"].set_index("fiscal_year")
    for year in FORECAST_HORIZON:
        w = fc.loc[year, "upper_95"] - fc.loc[year, "lower_95"]
        print(f"  {col:>9} FY{year}: point={fc.loc[year, 'point_forecast']:>14,.0f}  "
              f"width={w:>14,.0f}")
    print()

Known FY2027 Adopted (actual, not trend) for reference: 69,723,377

Interval width, FY2027 -> FY2031, by series (should widen monotonically):
    adopted FY2027: point=    76,266,259  width=   183,144,548
    adopted FY2028: point=    77,810,622  width=   192,011,451
    adopted FY2029: point=    79,354,986  width=   201,864,512
    adopted FY2030: point=    80,899,349  width=   212,566,642
    adopted FY2031: point=    82,443,713  width=   223,996,172

   modified FY2027: point=    84,913,305  width=   194,403,929
   modified FY2028: point=    88,381,022  width=   203,815,953
   modified FY2029: point=    91,848,739  width=   214,274,761
   modified FY2030: point=    95,316,456  width=   225,634,838
   modified FY2031: point=    98,784,173  width=   237,767,034

     actual FY2027: point=    71,178,054  width=   119,681,539
     actual FY2028: point=    73,991,103  width=   125,475,895
     actual FY2029: point=    76,804,153  width=   131,914,686
     actual FY2030: point=    79,617,

## Step 6: Chart -- 5-Year Forecast with Prediction Intervals

In [37]:
fig, ax = plt.subplots(figsize=(9, 6), facecolor=SURFACE)
ax.set_facecolor(SURFACE)

for col, color in SERIES.items():
    hist = train_table
    ax.plot(hist["fiscal_year"], hist[col] / 1e6, color=color, linewidth=2,
            marker="o", markersize=6, markerfacecolor=color,
            markeredgecolor=SURFACE, markeredgewidth=1.5, label=f"{col.capitalize()} (history)")

    fc = extended_fits[col]["forecast"].set_index("fiscal_year")
    fc_years = [hist["fiscal_year"].iloc[-1]] + FORECAST_HORIZON
    fc_points = [hist[col].iloc[-1] / 1e6] + list(fc["point_forecast"] / 1e6)
    fc_low = [hist[col].iloc[-1] / 1e6] + list(fc["lower_95"] / 1e6)
    fc_high = [hist[col].iloc[-1] / 1e6] + list(fc["upper_95"] / 1e6)

    ax.plot(fc_years, fc_points, color=color, linewidth=2, linestyle="--",
            marker="D", markersize=5, markerfacecolor=SURFACE, markeredgecolor=color)
    ax.fill_between(fc_years, fc_low, fc_high, color=color, alpha=0.12, linewidth=0)

ax.axvline(2026.5, color=BASELINE, linewidth=1, linestyle=":")
ax.text(2026.55, ax.get_ylim()[1] if ax.get_ylim()[1] else 0, "", fontsize=1)

ax.set_xlabel("Fiscal Year", color=INK_SECONDARY)
ax.set_ylabel("$ Millions", color=INK_SECONDARY)
ax.set_title(
    "TLC Budget Forecast: Next 5 Fiscal Years (FY2027-FY2031)\n"
    "Dashed = trend forecast, shaded = 95% prediction interval",
    color=INK_PRIMARY, fontsize=12, pad=12,
)
ax.set_xticks(list(train_table["fiscal_year"]) + FORECAST_HORIZON)
ax.tick_params(axis="x", labelrotation=45, colors=INK_MUTED)
ax.tick_params(axis="y", colors=INK_MUTED)

ax.grid(True, color=GRIDLINE, linewidth=1, linestyle="-")
ax.set_axisbelow(True)
for spine_name, spine in ax.spines.items():
    if spine_name in ("top", "right"):
        spine.set_visible(False)
    else:
        spine.set_color(BASELINE)

ax.legend(frameon=False, loc="upper left", labelcolor=INK_SECONDARY, fontsize=9)

fig.tight_layout()

FORECAST_PLOT_PATH = "tlc_forecast_5yr.png"
fig.savefig(FORECAST_PLOT_PATH, dpi=150, facecolor=SURFACE)
print(f"Saved 5-year forecast plot to {FORECAST_PLOT_PATH}")
plt.show()

Saved 5-year forecast plot to tlc_forecast_5yr.png


/var/folders/6v/95zgdc6s2w71rmzcwm6mww_40000gn/T/ipykernel_45860/84016564.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 7: Documented Limitations

- **~10 annual observations** (FY2017-2026) supports a simple linear trend
  with an honestly wide uncertainty band -- not a more flexible model. A
  polynomial/spline/ML regressor would fit noise in a 10-point series, not
  signal, and would report false confidence by construction.
- **FY2023's medallion-relief spike (\$155-162M vs. a \$54-66M neighboring
  norm) is left in the training data**, unlike DOT's FY2022 (which was
  dropped as a data-contamination artifact). It's real spending history, so
  dropping it would be cherry-picking the trend. But it does pull the fitted
  `adopted`/`modified` slopes upward relative to a version without it --
  meaning these forecasts likely run **slightly high** unless another
  one-time relief program recurs. Treat the point forecast as the
  upper-normal end of a plausible range, not dead center.
- **The forecast assumes historical growth continues.** It has no mechanism
  for policy shocks (a new medallion-relief-style appropriation, a TLC fee
  or regulatory change), macro shocks (a citywide budget cut), or
  operational shocks -- all of which have already moved this exact budget by
  more in one year (FY2023) than the trend moves in several.
- **The FY2027 sanity check (Step 4) is one data point**, not a validated
  track record. It says nothing about FY2028-2031 individually.
- **Practical use:** treat each point forecast as a planning anchor and the
  interval as the range worth budgeting flexibility around -- revisit as
  FY2027 actuals complete and FY2028 approaches, the same discipline used
  for the DOT forecast.